# 02 · Exploración descriptiva de cinco años

In [9]:
import os
import sys
from pathlib import Path

# Raíz del código: el notebook corre desde notebooks/. PROJECT_ROOT permite apuntar a otra carpeta
# de datos (por ejemplo, una copia sintética para validar el flujo).
CODE_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(CODE_ROOT))
os.chdir(os.environ.get("PROJECT_ROOT", CODE_ROOT))

import numpy as np
import pandas as pd

from src.config import load_validated_config

pd.set_option("display.max_columns", 30)
cfg = load_validated_config()
print("versión de la config:", cfg["version"])

versión de la config: v1


In [10]:
from src.stability import EDA_DIR, run_eda_five_years

eda = run_eda_five_years(cfg)
m5 = eda["matrix_five_years"]
pd.crosstab(m5["abc_class"], m5["xyz_class"].fillna("-"), margins=True)

xyz_class,X,Y,Z,All
abc_class,,,,
A,9,333,932,1274
B,1,208,719,928
C,1,160,686,847
All,11,701,2337,3049


Acuerdo y PSI de cada año frente al anterior y frente a la matriz de cinco años (descriptivo).

In [11]:
eda["stability_descriptive"]

,year,dimension,agreement_pct_vs_prev,agreement_pct_vs_5y,psi_vs_prev,psi_band
0,Y1,abc,NaN,74.479167,NaN,undefined
1,Y1,xyz,NaN,58.989599,NaN,undefined
2,Y2,abc,72.537879,78.455446,0.000563,stable
3,Y2,xyz,59.325397,60.556465,0.001453,stable
4,Y3,abc,73.346535,81.384083,0.002281,stable
5,Y3,xyz,60.016387,67.101551,0.010775,stable
6,Y4,abc,77.266436,78.058380,0.006096,stable
7,Y4,xyz,64.666431,63.185550,0.009693,stable
8,Y5,abc,81.108560,74.385044,0.000390,stable
9,Y5,xyz,62.331691,52.869793,0.048162,stable


Distribuciones por artículo: proporción de ceros en días activos, media frente a mediana, gasto y CV.

In [12]:
describe = eda["describe"]
describe.describe()

,active_days,zero_share,mean_daily_units,median_daily_units,total_units
count,3049.000000,3049.000000,3049.000000,3049.000000,3.049000e+03
mean,1539.762545,0.145817,13.025816,12.108560,2.096880e+04
std,393.005683,0.141786,24.460346,23.792703,4.289136e+04
min,415.000000,0.000549,0.354396,0.000000,5.930000e+02
25%,1269.000000,0.028022,3.220713,3.000000,4.320000e+03
50%,1820.000000,0.103198,6.260337,6.000000,9.369000e+03
75%,1820.000000,0.222822,13.026742,12.000000,2.074800e+04
max,1820.000000,0.781868,554.315934,556.000000,1.008855e+06


In [13]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

seg = cfg["segmentation"]
fig, axes = plt.subplots(2, 2, figsize=(11, 8))

# Curva ABC: artículos activos en orden de gasto (classify_abc ya los ordena), con los cortes.
active = m5[m5["exclusion_reason"] != "inactive_in_set"]
ax = axes[0, 0]
ax.plot(np.arange(1, len(active) + 1), active["cum_spend"] / active["spend"].sum())
for cut, label in zip(seg["abc_cuts"], ("A", "B"), strict=True):
    ax.axhline(cut, color="grey", linestyle="--", linewidth=1)
    ax.annotate(f"fin {label}", (0.0, cut), xycoords=("axes fraction", "data"),
                xytext=(4, 4), textcoords="offset points", ha="left")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set(title="Curva ABC (5 años)", xlabel="artículos ordenados por gasto",
       ylabel="gasto acumulado")

# CV en escala logarítmica (la cola de Z es larga), con los umbrales XYZ.
ax = axes[0, 1]
cv = m5["cv"].dropna()
cv = cv[cv > 0]
ax.hist(cv, bins=np.logspace(np.log10(cv.min()), np.log10(cv.max()), 50))
ax.set_xscale("log")
for thr in seg["xyz_thresholds"]:
    ax.axvline(thr, color="grey", linestyle="--", linewidth=1)
ax.set(title="CV de unidades (%)", xlabel="CV (escala log)", ylabel="artículos")

ax = axes[1, 0]
ax.hist(describe["zero_share"].dropna(), bins=50)
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.set(title="Proporción de ceros (días activos)", ylabel="artículos")

# Media frente a mediana: la diagonal y = x marca la simetría; symlog muestra las medianas en 0.
ax = axes[1, 1]
ax.scatter(describe["median_daily_units"], describe["mean_daily_units"], s=4, alpha=0.3)
top = np.nanmax(describe[["median_daily_units", "mean_daily_units"]].to_numpy())
ax.plot([0, top], [0, top], color="grey", linestyle="--", linewidth=1)
ax.set_xscale("symlog")
ax.set_yscale("symlog")
ax.set(title="Media frente a mediana diaria", xlabel="mediana", ylabel="media")

fig.tight_layout()
fig.savefig(Path(EDA_DIR) / "distributions.png", dpi=120)
plt.show()
plt.close(fig)

/tmp/ipykernel_106554/2697410118.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
